# 12 - Validation trade-off audit before another model

**CPU is sufficient; Run All.** Only reads the completed notebook-11 reports and their recorded unit paths. No image access, training, NBIS installation, new thresholds, checkpoint selection or test evaluation. Keep Drive mounted and preserve all referenced notebook-08/11 reports. You may keep the existing T4 runtime; this notebook does not need its GPU.

Notebook 11 improved wet_like FRR, but FAR sometimes increased, dry_like remained weak, and the shuffled control was competitive with curriculum. Audit BOTH policies already calculated in notebook 11: original-input fixed thresholds (operating-point shift), and condition/method-specific validation calibration to empirical conditional FMR <= 1% (SECONDARY diagnostic). The latter does NOT guarantee equal FMR or equal attempt FAR, is not independent test performance, and requires condition knowledge; never describe it as deployable blind calibration. Report achieved FAR/FMR alongside FRR/FNMR.

For every condition, compare input->curriculum, input->shuffled_control and shuffled_control->curriculum at both policies. Recompute all denominators and original-image recoveries/harms from the frozen pair scores. Include byte-distinct sensitivity at the same all-impression threshold; do not recalibrate on the sensitivity subset.

Uncertainty: 2000 paired bootstrap replicates, resampling scoped PROBE FINGER identities with replacement separately within each database, equal database weight. Use the same probe-finger draws for all methods, conditions and threshold policies. Within a finger include all probe impressions and gallery comparisons. Thresholds remain fixed; CIs do NOT include calibration uncertainty, checkpoint/model selection, seed variability, or dependence from shared gallery fingers. This is a conditional descriptive probe-cluster bootstrap, not a population-wide confidence guarantee. No biological subject IDs are available. No p-values, significance claims or best-method auto-selection; intervals are pointwise, NOT multiple-comparison corrected.

The synthetic summary averages nine conditions equally and six databases equally. Family summaries average three severities equally. Calculate aggregate intervals using the SAME draws, not by averaging interval endpoints. The same fingers appear across conditions, so treating condition results as independent would be wrong.

Verify the completed notebook-11 summary and every referenced unit CSV against their recorded checksums. Read-only archives are not rehashed here because only score/extraction CSVs are consumed. Compare the recomputed per-database rates to notebook-11 metrics before reporting. Any mismatch aborts; no dropping failed fingerprints or filling missing scores with zero.

Outputs: MyDrive/FingerprintResearch/analyses/m1_validation_tradeoff_v1/<configuration-hash>/. Completion marker: VALIDATION TRADEOFF AUDIT COMPLETE. Send the final tables to decide whether to pursue wet-like-focused work, add structure-preserving constraints, or abandon the curriculum hypothesis. These are exploratory validation results, not final Q3 novelty or held-out-test evidence.

In [ ]:
import hashlib
import inspect
import json
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
EVAL_DIR = PROJECT_ROOT / 'evaluations' / 'm1_controlled_validation_v1' / 'bef0c1363b1e6a721afc'
METHODS = ('input', 'm0', 'curriculum', 'shuffled_control')
COMPARISONS = [('input', 'curriculum'), ('input', 'shuffled_control'), ('shuffled_control', 'curriculum')]
POLICIES = ('original_fixed', 'condition_calibrated')
SEED, REPLICATES = 20261110, 2000

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))
print('CPU-only report audit. No images, GPU inference or NBIS subprocesses.')

In [ ]:
completion = json.loads((EVAL_DIR / 'completion.json').read_text(encoding='utf-8'))
configuration = json.loads((EVAL_DIR / 'configuration.json').read_text(encoding='utf-8'))
if (completion['status'] != 'm1_controlled_validation_matching_complete'
        or completion['configuration_sha256'] != file_digest(EVAL_DIR / 'configuration.json')
        or completion['variants_available'] != 40 or set(completion['methods']) != set(METHODS)):
    raise RuntimeError('Expected completed notebook-11 evaluation with all four methods.')
for name, expected in completion['file_sha256'].items():
    if file_digest(EVAL_DIR / name) != expected:
        raise RuntimeError(f'Completed notebook-11 summary changed: {name}')
reported = pd.read_csv(EVAL_DIR / 'validation_metrics.csv')
thresholds = pd.read_csv(EVAL_DIR / 'validation_thresholds.csv')
pixels = pd.read_csv(EVAL_DIR / 'pixel_and_failure_metrics.csv')
conditions = [f'{family}/{severity}' for family, severity in configuration['conditions']]
if len(conditions) != 10 or len(set(conditions)) != 10 or 'identity/none' not in conditions:
    raise RuntimeError('Expected ten unique validation conditions including identity.')
expected_units = {f'{family}_{severity}_{method}' for family, severity in configuration['conditions'] for method in METHODS}
if set(completion['unit_provenance']) != expected_units:
    raise RuntimeError('Completed unit provenance does not cover all forty variants.')
if thresholds.duplicated(['condition', 'method', 'dataset', 'db']).any():
    raise RuntimeError('Duplicate threshold records.')
units, extraction_frames, source_hash_rows = {}, [], []
canonical = None
for unit, provenance in sorted(completion['unit_provenance'].items()):
    directory = Path(provenance['directory'])
    marker_path = directory / 'completion.json'
    if file_digest(marker_path) != provenance['completion_sha256']:
        raise RuntimeError(f'Unit completion changed: {unit}')
    marker = json.loads(marker_path.read_text(encoding='utf-8'))
    expected_config = configuration['reference_evaluation_sha256'] if provenance['origin'] == 'notebook08' else completion['configuration_sha256']
    if marker['variant'] != unit or marker['configuration_sha256'] != expected_config:
        raise RuntimeError(f'Unit configuration/provenance mismatch: {unit}')
    for name in ('scores.csv', 'extraction.csv'):
        path = directory / name
        if file_digest(path) != marker['file_sha256'][name]:
            raise RuntimeError(f'Unit data changed: {path}')
        source_hash_rows.append({'unit': unit, 'file': name, 'sha256': marker['file_sha256'][name], 'path': str(path)})
    scored = pd.read_csv(directory / 'scores.csv')
    extraction = pd.read_csv(directory / 'extraction.csv')
    if scored['matchable'].dtype != bool or scored['duplicate_sensitivity_eligible'].dtype != bool:
        raise RuntimeError(f'Unexpected score booleans: {unit}')
    if scored['pair_id'].duplicated().any() or not set(scored['label']).issubset({0, 1}):
        raise RuntimeError(f'Invalid pair keys/labels: {unit}')
    valid_scores = scored.loc[scored['matchable'], 'score']
    if (not np.isfinite(valid_scores).all() or (valid_scores < 0).any()
            or not np.equal(valid_scores, np.floor(valid_scores)).all()
            or scored.loc[~scored['matchable'], 'score'].notna().any()):
        raise RuntimeError(f'Invalid score/failure encoding: {unit}')
    pair_columns = ['pair_id', 'dataset', 'db', 'label', 'probe_path', 'gallery_path',
                    'probe_identity', 'gallery_identity', 'duplicate_sensitivity_eligible']
    keys = scored[pair_columns].reset_index(drop=True)
    if canonical is None:
        canonical = keys
    elif not canonical.equals(keys):
        raise RuntimeError(f'Pair alignment differs across variants: {unit}')
    if len(scored) != completion['pairs_per_variant']:
        raise RuntimeError(f'Pair coverage differs: {unit}')
    condition, method = provenance['condition'], provenance['method']
    if not extraction['condition'].eq(condition).all() or not extraction['method'].eq(method).all():
        raise RuntimeError(f'Extraction method/condition mismatch: {unit}')
    units[(condition, method)] = scored
    extraction_frames.append(extraction.assign(audit_condition=condition, audit_method=method))
databases = sorted(set(zip(canonical['dataset'], canonical['db'])))
if len(databases) != 6 or set(canonical['dataset']) != {'FVC2000', 'FVC2002'}:
    raise RuntimeError('Expected six source-only validation databases.')
print('Verified score variants:', len(units), '| Pairs per variant:', len(canonical))
print('No archives, images, train or test samples were opened.')

In [ ]:
def rates(frame, threshold):
    if not np.isfinite(threshold) or threshold < 0 or threshold != int(threshold):
        raise ValueError('Expected a nonnegative integer frozen threshold.')
    genuine, impostor = frame['label'].eq(1), frame['label'].eq(0)
    if not genuine.any() or not impostor.any():
        raise RuntimeError('Need genuine and impostor attempts.')
    accepted = frame['matchable'] & frame['score'].ge(threshold)
    vg, vi = genuine & frame['matchable'], impostor & frame['matchable']
    return {'genuine_attempts': int(genuine.sum()), 'impostor_attempts': int(impostor.sum()),
        'unmatchable_genuine': int((genuine & ~frame['matchable']).sum()),
        'unmatchable_impostor': int((impostor & ~frame['matchable']).sum()),
        'attempt_frr': float((~accepted[genuine]).mean()), 'attempt_far': float(accepted[impostor].mean()),
        'conditional_fnmr': float((~accepted[vg]).mean()) if vg.any() else None,
        'conditional_fmr': float(accepted[vi].mean()) if vi.any() else None}

def percentile_interval(values):
    values = np.asarray(values, dtype=float)
    if values.shape != (REPLICATES,) or not np.isfinite(values).all():
        raise ValueError('Expected finite paired bootstrap replicates.')
    return np.quantile(values, [0.025, 0.975]).tolist()

def bootstrap_pair(before, after, before_thresholds, after_thresholds):
    result = {}
    if before['pair_id'].tolist() != after['pair_id'].tolist():
        raise RuntimeError('Bootstrap comparisons must be pair aligned.')
    for label, rate in ((1, 'frr'), (0, 'far')):
        db_points, db_replicates = [], []
        for dataset, db in databases:
            selected = before['dataset'].eq(dataset) & before['db'].eq(db) & before['label'].eq(label)
            left, right = before.loc[selected], after.loc[selected]
            if left.empty:
                raise RuntimeError('Missing database/label attempts in bootstrap.')
            left_accept = left['matchable'] & left['score'].ge(before_thresholds[(dataset, db)])
            right_accept = right['matchable'] & right['score'].ge(after_thresholds[(dataset, db)])
            delta = right_accept.astype(float) - left_accept.astype(float)
            if label == 1:
                delta = -delta
            counts = pd.DataFrame({'identity': left['probe_identity'], 'delta': delta}).groupby('identity', sort=True).agg(
                delta_sum=('delta', 'sum'), attempts=('delta', 'size')).reindex(finger_keys[(dataset, db)])
            if counts.isna().any().any():
                raise RuntimeError('Incomplete probe-finger coverage in bootstrap.')
            weights = bootstrap_weights[(dataset, db)]
            numerator = weights @ counts['delta_sum'].to_numpy(dtype=float)
            denominator = weights @ counts['attempts'].to_numpy(dtype=float)
            if (denominator <= 0).any():
                raise RuntimeError('Bootstrap replicate has no attempts.')
            db_replicates.append(numerator / denominator)
            db_points.append(float(counts['delta_sum'].sum() / counts['attempts'].sum()))
        values = np.mean(db_replicates, axis=0)
        low, high = percentile_interval(values)
        result[rate] = {'delta': float(np.mean(db_points)), 'low': low, 'high': high, 'replicates': values}
    return result

def interval_row(scope, policy, baseline, method, values):
    row = {'scope': scope, 'threshold_policy': policy, 'baseline_method': baseline, 'method': method,
           'bootstrap_replicates': REPLICATES, 'ci_type': 'pointwise conditional probe-finger percentile; fixed thresholds'}
    for rate in ('frr', 'far'):
        low, high = percentile_interval(values[rate]['replicates'])
        row.update({f'delta_{rate}_pp': 100 * values[rate]['delta'],
                    f'delta_{rate}_ci_low_pp': 100 * low, f'delta_{rate}_ci_high_pp': 100 * high})
    return row

In [ ]:
finger_keys, bootstrap_weights = {}, {}
for dataset, db in databases:
    keys = sorted(canonical.loc[canonical['dataset'].eq(dataset) & canonical['db'].eq(db), 'probe_identity'].unique())
    if len(keys) < 2:
        raise RuntimeError('Need at least two probe-finger clusters per database.')
    finger_keys[(dataset, db)] = keys
    seed = int.from_bytes(hashlib.sha256(f'{SEED}|probe_cluster|{dataset}|{db}'.encode()).digest()[:8], 'big')
    bootstrap_weights[(dataset, db)] = np.random.default_rng(seed).multinomial(len(keys), np.full(len(keys), 1 / len(keys)), size=REPLICATES)
helper_source = '\n\n'.join(inspect.getsource(function) for function in (rates, percentile_interval, bootstrap_pair, interval_row))
analysis_config = {'version': 'm1_validation_tradeoff_v1', 'evaluation_configuration_sha256': completion['configuration_sha256'],
    'evaluation_completion_sha256': file_digest(EVAL_DIR / 'completion.json'), 'seed': SEED, 'replicates': REPLICATES,
    'policies': POLICIES, 'comparisons': COMPARISONS, 'methods': METHODS,
    'bootstrap': 'paired probe-finger clusters within database; equal database weights; same draws across conditions; fixed thresholds',
    'limitations': 'conditional on shared fixed galleries, calibration and selected checkpoints; pointwise intervals, not multiplicity corrected',
    'numpy_version': np.__version__, 'pandas_version': pd.__version__,
    'helper_sha256': hashlib.sha256(helper_source.encode()).hexdigest(),
    'draw_sha256': {f'{dataset}_{db}': hashlib.sha256(weights.astype('<i8').tobytes()).hexdigest()
                    for (dataset, db), weights in bootstrap_weights.items()}}
analysis_data = json_bytes(analysis_config)
analysis_sha = hashlib.sha256(analysis_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'analyses' / 'm1_validation_tradeoff_v1' / analysis_sha[:20]
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': analysis_data, 'analysis_helpers.py.txt': helper_source.encode()}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen analysis changed: {path}')
    if not path.exists():
        write_verified(path, data)
write_csv(REPORT_DIR / 'consumed_unit_files.csv', pd.DataFrame(source_hash_rows))
print('Analysis frozen:', analysis_sha)
print('Outputs:', REPORT_DIR)

In [ ]:
rate_rows, harms, threshold_lookup = [], [], {}
for condition in conditions:
    for method in METHODS:
        scored = units[(condition, method)]
        for policy in POLICIES:
            lookup = {}
            for (dataset, db), frame in scored.groupby(['dataset', 'db'], sort=True):
                saved = thresholds[thresholds['condition'].eq(condition) & thresholds['method'].eq(method)
                                   & thresholds['dataset'].eq(dataset) & thresholds['db'].eq(db)]
                if len(saved) != 1:
                    raise RuntimeError('Missing/duplicate frozen threshold record.')
                threshold = saved.iloc[0]['original_fixed_threshold' if policy == 'original_fixed' else 'condition_calibrated_threshold']
                if pd.isna(threshold):
                    raise RuntimeError(f'Unavailable frozen calibration: {condition}/{method}/{dataset}/{db}; do not invent a threshold.')
                lookup[(dataset, db)] = threshold
                for view, selected in (('all_impressions', frame), ('byte_distinct_sensitivity', frame[frame['duplicate_sensitivity_eligible']])):
                    values = rates(selected, threshold)
                    record = reported[reported['condition'].eq(condition) & reported['method'].eq(method)
                        & reported['dataset'].eq(dataset) & reported['db'].eq(db)
                        & reported['threshold_policy'].eq(policy) & reported['view'].eq(view)]
                    if len(record) != 1 or record.iloc[0]['threshold'] != threshold:
                        raise RuntimeError('Missing/duplicate or inconsistent notebook-11 metric row.')
                    for name, value in values.items():
                        previous = record.iloc[0][name]
                        if (value is None and not pd.isna(previous)) or (value is not None and (not np.isfinite(previous) or abs(previous - value) > 1e-10)):
                            raise RuntimeError(f'Recomputed notebook-11 metric mismatch: {condition}/{method}/{dataset}/{db}/{name}')
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] is not None and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Frozen calibration exceeds empirical conditional FMR 1%.')
                    rate_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, 'threshold': threshold, **values})
            threshold_lookup[(condition, method, policy)] = lookup
for condition in conditions:
    for policy in POLICIES:
        fixed = [threshold_lookup[(condition, method, policy)] for method in METHODS]
        if policy == 'original_fixed' and any(value != fixed[0] for value in fixed[1:]):
            raise RuntimeError('Original fixed thresholds differ across methods.')
        if policy == 'original_fixed' and fixed[0] != threshold_lookup[('identity/none', 'input', policy)]:
            raise RuntimeError('Original fixed thresholds differ across conditions.')
for baseline, method in COMPARISONS:
    left, right = units[('identity/none', baseline)], units[('identity/none', method)]
    for key in databases:
        dataset, db = key
        selected = left['dataset'].eq(dataset) & left['db'].eq(db) & left['label'].eq(1)
        threshold = threshold_lookup[('identity/none', 'input', 'original_fixed')][key]
        before = left.loc[selected, 'matchable'] & left.loc[selected, 'score'].ge(threshold)
        after = right.loc[selected, 'matchable'] & right.loc[selected, 'score'].ge(threshold)
        harms.append({'baseline_method': baseline, 'method': method, 'dataset': dataset, 'db': db,
            'genuine_attempts': int(selected.sum()), 'recoveries': int((~before & after).sum()), 'harms': int((before & ~after).sum())})
recomputed = pd.DataFrame(rate_rows)
write_csv(REPORT_DIR / 'recomputed_database_metrics.csv', recomputed)
write_csv(REPORT_DIR / 'original_image_harms.csv', pd.DataFrame(harms))
macro = recomputed.groupby(['condition', 'method', 'threshold_policy', 'view'], sort=True).agg(
    databases=('db', 'size'), attempt_frr=('attempt_frr', 'mean'), attempt_far=('attempt_far', 'mean'),
    conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Macro summaries must cover all six databases.')
write_csv(REPORT_DIR / 'macro_operating_points.csv', macro)
print('All recomputed rates agree with frozen notebook-11 metrics.')

In [ ]:
interval_rows, replicate_results = [], {}
for condition in conditions:
    for policy in POLICIES:
        for baseline, method in COMPARISONS:
            values = bootstrap_pair(units[(condition, baseline)], units[(condition, method)],
                threshold_lookup[(condition, baseline, policy)], threshold_lookup[(condition, method, policy)])
            replicate_results[(condition, policy, baseline, method)] = values
            interval_rows.append(interval_row(condition, policy, baseline, method, values))
    print('Paired probe-finger bootstrap:', condition, flush=True)
synthetic_conditions = [condition for condition in conditions if condition != 'identity/none']
scopes = {'synthetic_nine_equal_weight': synthetic_conditions,
          **{f'{family}_three_severities': [condition for condition in conditions if condition.startswith(family + '/')]
             for family in ('dry_like', 'wet_like', 'generic')}}
for scope, members in scopes.items():
    if len(members) != (9 if scope == 'synthetic_nine_equal_weight' else 3):
        raise RuntimeError('Unexpected aggregate condition coverage.')
    for policy in POLICIES:
        for baseline, method in COMPARISONS:
            pieces = [replicate_results[(condition, policy, baseline, method)] for condition in members]
            values = {rate: {'delta': float(np.mean([piece[rate]['delta'] for piece in pieces])),
                      'replicates': np.mean([piece[rate]['replicates'] for piece in pieces], axis=0)} for rate in ('frr', 'far')}
            interval_rows.append(interval_row(scope, policy, baseline, method, values))
intervals = pd.DataFrame(interval_rows)
write_csv(REPORT_DIR / 'paired_probe_finger_intervals.csv', intervals)
aggregate_rows = []
for scope, members in scopes.items():
    subset = macro[macro['condition'].isin(members)]
    grouped = subset.groupby(['method', 'threshold_policy', 'view'], sort=True).agg(
        conditions=('condition', 'size'), attempt_frr=('attempt_frr', 'mean'), attempt_far=('attempt_far', 'mean'),
        conditional_fnmr=('conditional_fnmr', 'mean'), conditional_fmr=('conditional_fmr', 'mean')).reset_index()
    if not grouped['conditions'].eq(len(members)).all():
        raise RuntimeError('Aggregate operating points have incomplete conditions.')
    aggregate_rows.append(grouped.assign(scope=scope))
aggregates = pd.concat(aggregate_rows, ignore_index=True)
write_csv(REPORT_DIR / 'aggregate_operating_points.csv', aggregates)
print('Deltas = method minus baseline, in percentage points; negative FRR is better, positive FAR is worse.')
print(intervals[intervals['scope'].eq('synthetic_nine_equal_weight')].to_string(index=False))

In [ ]:
all_extraction = pd.concat(extraction_frames, ignore_index=True)
pixel_summary = all_extraction.groupby(['audit_condition', 'audit_method'], sort=True).agg(
    images=('relative_path', 'size'), float_input_mae=('float_input_mae', 'mean'),
    float_output_mae=('float_output_mae', 'mean'), png_input_mae=('png_input_mae', 'mean'),
    png_output_mae=('png_output_mae', 'mean'), changed_pixel_fraction=('changed_pixel_fraction', 'mean'),
    identical_image_fraction=('png_identical_to_input', 'mean')).reset_index()
write_csv(REPORT_DIR / 'image_weighted_pixel_summary.csv', pixel_summary)
print('\nSecondary diagnostic: separately calibrated empirical conditional FMR <= 1%, NOT necessarily equal achieved FMR/FAR.')
print(aggregates[aggregates['scope'].eq('synthetic_nine_equal_weight') & aggregates['view'].eq('all_impressions')].to_string(index=False))
print('\nOriginal-image recoveries/harms at the SAME original thresholds:')
print(pd.DataFrame(harms).groupby(['baseline_method', 'method'])[['recoveries', 'harms']].sum().to_string())
print('\nFamily-specific paired changes (pointwise, conditional intervals; not significance tests):')
print(intervals[intervals['scope'].str.endswith('three_severities') & intervals['threshold_policy'].eq('condition_calibrated')].to_string(index=False))
files = ['configuration.json', 'analysis_helpers.py.txt', 'consumed_unit_files.csv', 'recomputed_database_metrics.csv',
    'original_image_harms.csv', 'macro_operating_points.csv', 'paired_probe_finger_intervals.csv',
    'aggregate_operating_points.csv', 'image_weighted_pixel_summary.csv']
marker = {'status': 'validation_tradeoff_audit_complete', 'configuration_sha256': analysis_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files},
    'evaluation_completion_sha256': file_digest(EVAL_DIR / 'completion.json'),
    'training_performed': False, 'new_thresholds_calibrated': False, 'test_evaluated': False,
    'independent_test_improvement_verified': False,
    'limitations': analysis_config['limitations']}
write_verified(REPORT_DIR / 'completion.json', json_bytes(marker))
print('VALIDATION TRADEOFF AUDIT COMPLETE:', REPORT_DIR)
print('Send aggregate operating points and paired intervals before selecting another architecture or claiming curriculum novelty.')